# Pipeline layer review

Read-only look at what the jobs wrote, using **pandas** rather than Spark -- no
JVM, no session startup, no Iceberg jars. Loads in a second or two.

The path logic lives in `review_layers.py` beside this notebook, which derives
every path from `config/local.yaml` through the pipeline's own `build_path`. So
this notebook cannot drift from where the jobs actually write.

**Read-only by design.** Nothing here writes, deletes or moves an S3 object.

In [1]:
import sys
from datetime import datetime

import pandas as pd

sys.path.insert(0, "..")   # the python_scripts package, from the repo root
sys.path.insert(0, ".")    # review_layers.py, beside this notebook

import review_layers as rl

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

cfg, fs = rl.connect("prod")
print(f"bucket: {cfg.bucket}   env: {cfg.env}   region: {cfg.region}")
print(f"tables: {sorted(cfg.tables)}")

bucket: globalpartners-project-cjk   env: prod   region: us-east-1
tables: ['date_dim', 'order_item_options', 'order_items']


## 1. What exists

Start here after any run. One row per prefix holding parquet, so an empty
result means the job wrote nothing -- and a suspiciously small `bytes` means it
wrote an empty frame: schema only, no rows.

In [2]:
rl.list_prefixes(cfg, fs)

,prefix,files,bytes
0,globalpartners-project-cjk/prod/raw/date_dim,1,5300
1,globalpartners-project-cjk/prod/raw/order_item...,1,5526384
2,globalpartners-project-cjk/prod/raw/order_item...,1,11725744
3,globalpartners-project-cjk/prod/raw/order_item...,1,2612
4,globalpartners-project-cjk/prod/warehouse/silv...,4,14328
5,globalpartners-project-cjk/prod/warehouse/silv...,2,6644717
6,globalpartners-project-cjk/prod/warehouse/silv...,2,11357
7,globalpartners-project-cjk/prod/warehouse/silv...,2,27310
8,globalpartners-project-cjk/prod/warehouse/silv...,2,55847
9,globalpartners-project-cjk/prod/warehouse/silv...,2,75148


## 2. Was anything held back?

Reason counts across both stages and every table. This is the health check --
run it before trusting a batch.

`raw_ingest` can only reject a null business key. Cast failures need a target
type to try, so they can only surface in `raw_to_silver`.

In [3]:
rl.reject_summary(cfg, fs)

,_dataset,_quarantine_stage,_reject_reason,rows


## 3. The held rows themselves

With their **original** values, because the split runs before any conversion.
A transformed copy would be useless for working out what the source contained.

`batch_date=None` reads every batch at once; pass a date to narrow it.

In [ ]:
held = rl.read_quarantine(cfg, fs, "order_items", "raw_ingest")

print(f"{len(held)} row(s) held")
held

In [ ]:
# Just the columns that matter for triage
triage = ["ORDER_ID", "LINEITEM_ID", "ITEM_PRICE", "_reject_reason",
          "_quarantine_stage", "_quarantine_table", "_batch_date"]

held[triage] if not held.empty else "nothing held"

In [ ]:
# The other stage. Empty until raw_to_silver has run.
rl.read_quarantine(cfg, fs, "order_items", "raw_to_silver")

## 4. Raw layer

A faithful copy of the source, partitioned by **when rows arrived** (the batch
date) rather than when the orders were created.

In [6]:
raw = rl.read_raw(cfg, fs, "order_items", datetime(2026, 10, 16))

print(f"{len(raw):,} rows")
raw.head()

203,518 rows


,APP_NAME,RESTAURANT_ID,CREATION_TIME_UTC,ORDER_ID,USER_ID,PRINTED_CARD_NUMBER,IS_LOYALTY,CURRENCY,LINEITEM_ID,ITEM_CATEGORY,ITEM_NAME,ITEM_PRICE,ITEM_QUANTITY,inserted_at
0,Alltown Fresh,63bc98a7519adc105105a990,2023-03-08T11:03:32.223Z,64086b8463905bf310002079,63ee144950286a8367041911,NaN,False,USD,64086b84d5fbd8718a0717e6,Breakfast,Bacon Egg & Cheese*,5.99,1,2026-10-16 21:03:39.089
1,Alltown Fresh,5f6a6c1537ab46bd38e9df75,2023-05-13T12:45:00.475Z,645f864c1c4b0312f10cfa75,NaN,NaN,False,USD,645f869568ab8e534e099ea7,Breakfast,Breakfast Burrito*,7.99,1,2026-10-16 21:03:39.089
2,Alltown Fresh,5fece84aadb0d51509c36f22,2023-02-03T07:15:28.519Z,63dcb490665d80c6be09a618,5ece77fe902ad501337b23fd,NaN,False,USD,63dcb49b665d80c6be09a657,Breakfast,Bacon Egg & Cheese*,5.99,1,2026-10-16 21:03:39.089
3,Alltown Fresh,5f6a6c1537ab46bd38e9df75,2023-08-17T16:18:02.231Z,64de483a7028adc2520d8cf0,NaN,NaN,False,USD,64de484eb7b035d89d004141,Salads,Greek Salad,11.99,1,2026-10-16 21:03:39.089
4,Alltown Fresh,622107b40ac81503e0369ca6,2023-08-20T16:02:12.518Z,64e23904baef31a741016be9,NaN,NaN,False,USD,64e2395465826c9b650c1c37,Breakfast,Breakfast Burrito*,5.99,1,2026-10-16 21:03:39.089


In [4]:
# date_dim is load_type: static, so it has no batch partition -- omit the date.
rl.read_raw(cfg, fs, "date_dim").head()

,date_key,year,month,week,day_of_week,is_weekend,is_holiday,holiday_name
0,2023-01-01,2023,1,52,Sunday,True,True,New Year's Day
1,2023-01-02,2023,1,1,Monday,False,True,New Year's Day Observed
2,2023-01-03,2023,1,1,Tuesday,False,False,NaN
3,2023-01-04,2023,1,1,Wednesday,False,False,NaN
4,2023-01-05,2023,1,1,Thursday,False,False,NaN


## 5. Checks worth running after a load

Cheap questions that catch the failure modes this pipeline actually has.

In [7]:
# Does the watermark line up with the --period you passed? Every row of an
# initial load carries the single instant the ALTER TABLE ran.
raw["inserted_at"].agg(["min", "max"])

min   2026-10-16 21:03:39.089
max   2026-10-16 21:03:39.089
Name: inserted_at, dtype: datetime64[ns]

In [ ]:
                                    # Will CREATION_TIME_UTC survive the silver cast? A non-ISO value gets
# quarantined, so an unexpected format here means an empty silver table.
parsed = pd.to_datetime(raw["CREATION_TIME_UTC"], format="ISO8601",
                        errors="coerce", utc=True)

print(f"unparseable: {parsed.isna().sum()} of {len(parsed):,}")
print(f"event-time range: {parsed.min()}  ->  {parsed.max()}")

unparseable: 0 of 203,518
event-time range: 2020-04-21 18:56:44.244000+00:00  ->  2024-02-21 02:52:35.366000+00:00


In [ ]:
# Are the merge keys unique? Duplicates are quarantined rather than merged, so
# a surprise here means rows you expected in silver are being held instead.
keys = cfg.get_table_config("order_items").business_keys
group_sizes = raw.groupby(keys).size()

print(f"duplicate key groups: {(group_sizes > 1).sum()}")
group_sizes[group_sizes > 1].head()

In [ ]:
# Null business keys -- these are what raw_ingest quarantines.
raw[keys].isna().sum()

## A note on the silver layer

Silver is **not** in S3 when you run `--env local`. `create_spark_session`
configures the Iceberg catalog with `warehouse = file:///<repo>/warehouse`, so
silver tables land on local disk while raw goes to S3.

That is also why pandas cannot read silver the way it reads the layers above.
An Iceberg table is a set of parquet files *plus metadata deciding which of
them are live*; reading the parquet directly would show rows that later
snapshots deleted.

To query silver or gold, use Spark -- see **`review_gold.ipynb`** beside this
notebook, which does exactly that:

```python
spark.table("iceberg.silver.order_items")
spark.table("iceberg.gold.daily_restaurant_sales")
```

The catalog is named `iceberg` and serves every Iceberg layer; the namespace
is the layer name from `layers:` in config. `pyiceberg` pointed at the
warehouse would also work and starts far faster, at the cost of another
dependency.